# 02. COSEBIs and the KiDS-Legacy validation

* **COSEBIs validation** (`docs/COSEBIS.md`): E_n modes from the KV450 xi+/- best fit, the COSEBIs-only MCMC and the re-weighting of the xi+/- chain.
* **KiDS-Legacy validation** (`docs/KIDS_LEGACY_VALIDATION.md`): the `kidspipe.legacy` COSEBIs model against the official KiDS-Legacy chain (3509 chain rows, seed 7).

Both read the KV450 results from `$KIDSPIPE_RUNS/kv450_final` when you produced them in notebook 01, otherwise from the copy shipped in the repository, so this notebook runs on its own.

**Acknowledgement.** The DR5 / KiDS-Legacy data are KiDS products. Acknowledge KiDS and cite Wright et al. 2024 (A&A 686, A170) and the KiDS-Legacy papers
(<https://kids.strw.leidenuniv.nl/DR5/legacy_wl.php>); the full acknowledgement statement is on that KiDS page (the repository's `README.md` and `data/README.md` only summarise it). The repository is private; do not share the data or results
outside the terms of the KiDS release.

### Which runtime?

| | free Colab | Colab Pro / high-RAM |
|---|---|---|
| vCPU | 2 | about 8 |
| RAM | 12.7 GB | about 51 GB |
| session length | about 12 h (idle and background limits apply) | up to about 24 h |
| local disk | about 100 GB (wiped at the end of the session) | the same |
| Google Drive | persistent, shared 15 GB quota (free) | persistent |

The notebooks use `os.cpu_count()` for threads and worker processes. All times in these notebooks are **estimates** scaled from timings recorded on a
22-core development workstation (WSL2): linear scaling in cores is assumed and a Colab vCPU is assumed to be as fast as one of its cores, which is probably
optimistic. Treat them as lower bounds on a shared, noisy Colab host.

### One-time preparation: access to the private repository

The repository is private, so the notebook needs a GitHub **fine-grained personal access token**. The token is stored in **Colab Secrets**
and is never written into a cell, a file or the output.

1. GitHub, Settings, Developer settings, Personal access tokens, **Fine-grained tokens**, Generate new token. Resource owner: the account that owns the
   repository. Repository access: **Only select repositories**, choose this repository. Permissions: **Contents: Read-only** (Metadata is added
   automatically). Pick a short expiry. Copy the token once.
2. In Colab, left sidebar, the key icon (**Secrets**), Add new secret: name `GH_TOKEN`, value = the token, switch **Notebook access** on for this notebook
   (the switch is per notebook).
3. Edit `GITHUB_REPO` in the setup cell below (`owner/name`).

No GitHub access? Put a zip of the repository at `MyDrive/kidspipe/kidspipe_repo.zip`; the setup cell unpacks it instead of cloning.

## Session setup

A Colab runtime is fresh in every session: **run the next cell first, every time**. It mounts Drive, clones or updates the repository, installs the package if needed and sets the environment variables (`KIDSPIPE_RUNS`, `KIDSPIPE_DR5_WORK`, `KIDSPIPE_DR5_CATALOGUE`, `KIDSPIPE_PLOTS`, thread counts, numba cache on Drive) described in `docs/COLAB.md`.

In [ ]:
#@title 1. Session setup (run this first, in EVERY session) { display-mode: "form" }
#@markdown Private repository to clone (`owner/name`); the token comes from the Colab Secret `GH_TOKEN`.
GITHUB_REPO = "OWNER/REPO"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
#@markdown Persistent outputs live on Drive; the 16 GB catalogue lives on the local disk (never on Drive).
DRIVE_ROOT = "/content/drive/MyDrive/kidspipe"  #@param {type:"string"}
CATALOGUE_DIR = "/content/dr5"  #@param {type:"string"}
REPO_DIR = "/content/kidspipe"

import base64, importlib, os, pathlib, shutil, subprocess, sys

def _git(args, token=None, cwd=None):
    """git, with the token passed as an HTTP header (not stored in .git/config); the token is masked in anything returned."""
    cmd, secret = ["git"], []
    if token:
        b64 = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        cmd += ["-c", f"http.extraheader=Authorization: Basic {b64}"]
        secret = [token, b64]
    r = subprocess.run(cmd + args, cwd=cwd, capture_output=True, text=True)
    out = r.stdout + r.stderr
    for s in secret:
        out = out.replace(s, "***")
    return r.returncode, out.strip()

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: Drive is not mounted; DRIVE_ROOT must be a writable directory.")

token = None
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception as e:                                   # secret missing, or notebook access not granted
    print(f"No GH_TOKEN secret available ({type(e).__name__}); using an existing clone or the Drive zip if there is one.")

repo = pathlib.Path(REPO_DIR)
url = f"https://github.com/{GITHUB_REPO}.git"
if (repo / "pyproject.toml").exists():
    rc, out = _git(["fetch", "--depth", "1", "origin", BRANCH], token, cwd=repo)
    if rc == 0:
        _git(["reset", "--hard", "FETCH_HEAD"], cwd=repo)
        print("repository updated to the latest", BRANCH)
    else:
        print("could not update (using the existing clone):", out[-300:])
elif token or not pathlib.Path(DRIVE_ROOT, "kidspipe_repo.zip").exists():
    if GITHUB_REPO.startswith("OWNER/"):
        raise SystemExit("Edit GITHUB_REPO (owner/name) in this cell, and add the GH_TOKEN secret (see the markdown above).")
    rc, out = _git(["clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR], token)
    if rc != 0:
        raise SystemExit("git clone failed (wrong GITHUB_REPO, or the token lacks Contents: read access?):\n" + out[-500:])
    print("cloned", GITHUB_REPO)
else:
    tmp = pathlib.Path("/content/_unzip"); shutil.rmtree(tmp, ignore_errors=True)
    shutil.unpack_archive(str(pathlib.Path(DRIVE_ROOT, "kidspipe_repo.zip")), str(tmp))
    top = [p for p in tmp.iterdir() if p.is_dir()]
    shutil.move(str(top[0]) if len(top) == 1 and not (tmp / "pyproject.toml").exists() else str(tmp), REPO_DIR)
    print("unpacked the repository from the Drive zip")

sys.path.insert(0, f"{REPO_DIR}/scripts/colab")
import setup_env; importlib.reload(setup_env)
INFO = setup_env.setup(REPO_DIR, DRIVE_ROOT, CATALOGUE_DIR)      # env vars, pip install -e '.[all]' if needed, versions
os.chdir(REPO_DIR)
WORKERS = os.cpu_count() or 1

### Expected wall times (ESTIMATES, scaled from the 22-core workstation)

| step | core-h | wall, 2 vCPU | wall, 8 vCPU |
|---|---|---|---|
| `cosebis_validate.py main` (plots 01-05, 07; numbers) | 0.1 | 5-10 min | 2 min |
| `cosebis_mcmc.py 5 10 ...` (10 chains, 5 modes) | 5 | 2.1 h | 0.5 h |
| `cosebis_validate.py post` (re-weighting, plot 06) | about 1 | 0.5 h | 10 min |
| `pytest tests/test_cosebis.py` | | 5 min | 2 min |
| `legacy_validate.py eval` (3509 samples, resumable) | 3.5 s/sample/core: about 3.4 | 1.8-2 h | 35-45 min |
| `legacy_validate.py plots`, `timing` | small | 5 min | 2 min |

In [ ]:
import os, pathlib, shutil
from setup_env import stage, sh
W = WORKERS
RUNS = pathlib.Path(os.environ["KIDSPIPE_RUNS"]); PLOTS = pathlib.Path(os.environ["KIDSPIPE_PLOTS"])
S = "python3 scripts"
T4 = f"OMP_NUM_THREADS={min(W, 4)} NUMBA_NUM_THREADS={min(W, 4)}"     # the validation scripts: a few threads
T1 = "OMP_NUM_THREADS=1 NUMBA_NUM_THREADS=1"                          # worker-process steps: one thread per process
def show(*names):
    from IPython.display import Image, display
    for n in names:
        p = PLOTS / n
        print(p) if p.exists() else print("missing:", p)
        if p.exists(): display(Image(filename=str(p)))

## Part A. COSEBIs

In [ ]:
stage('cosebis_main', f'{T4} {S}/cosebis_validate.py main')
show('cosebis_03_routes.png', 'cosebis_04_data_vs_model.png')

The COSEBIs-only MCMC: one process per chain, `COSEBIS_WALL` = wall-clock limit **per chain** in seconds. The documented run is 10 chains x 1500 s on a machine with about 2 effective
cores per chain-group; the cell scales the limit by chains / min(chains, vCPU) so that each chain gets the same CPU time. Writes `<runs>/cosebis/chain_n5_c*.npz`.

In [ ]:
NCH = 10
wall = round(1500 * NCH / min(NCH, W))
stage('cosebis_mcmc', f'COSEBIS_WALL={wall} {S}/cosebis_mcmc.py 5 {NCH} 10000 1500 n5')
stage('cosebis_post', f'{T4} {S}/cosebis_validate.py post')
show('cosebis_06_posterior.png')

In [ ]:
!python -m pytest -q -p no:cacheprovider tests/test_cosebis.py

## Part B. KiDS-Legacy validation

`eval` evaluates the COSEBIs likelihood at 3509 samples of the official chain with one process per worker (one thread each) and **resumes** from `<runs>/legacy_val/chi2.npz`: if the
session ends, run the cell again. The first call builds the W_n operator tables (10-20 s each) into `<runs>/legacy/cache`.

In [ ]:
rc = sh(f'{T1} {S}/legacy_validate.py eval --workers {W}')
print('exit status', rc, '(re-run this cell to resume if it did not finish)')

In [ ]:
if rc == 0:
    stage('legacy_plots', f'{T4} {S}/legacy_validate.py plots')
    stage('legacy_timing4', f'OMP_NUM_THREADS={min(W, 4)} {S}/legacy_validate.py timing')
    stage('legacy_timing1', f'OMP_NUM_THREADS=1 {S}/legacy_validate.py timing')
    show('legacy_01_chi2_vs_official.png', 'legacy_03_S8_official_vs_ours.png')

Optional IA-scale diagnostic (reads `legacy_val/chi2.npz` and `selection.npz`; must run from the repository root):

In [ ]:
RUN_DIAG = False   #@param {type:"boolean"}
if RUN_DIAG:
    sh(f'OMP_NUM_THREADS={min(W, 4)} python3 scripts/legacy_ia_scale_diag.py 200', cwd=REPO_DIR)

### Compare with the shipped reference results

In [ ]:
import json
mine, ref = RUNS / "legacy_val" / "summary.json", pathlib.Path(REPO_DIR) / "runs" / "legacy_val" / "summary.json"
for name, p in (("yours", mine), ("shipped", ref)):
    print(name, ":", (json.dumps(json.load(open(p)))[:400] + " ...") if p.exists() else "(none yet)")

### Keep the figures

The scripts write to `$KIDSPIPE_PLOTS` (on Drive). The repository's own `plots/` keeps the shipped reference figures untouched.

In [ ]:
print(sorted(p.name for p in PLOTS.glob('*.png')))